# Project 6 — Notebook 21: Business Summary & Recommendations
### Infrastructure Stress Analysis · PM Priority Engine · Targeted Site Interventions

---

| | |
|---|---|
| **Scope** | NCR (Region 3) · Infrastructure stress profiling · PM priority scoring |
| **Feeds from** | NB19 (Infrastructure Stress Analysis) · NB20 (PM Priority Engine) |
| **Audience** | Operations Leadership / PM Planning Team / Zone Managers |

---

### What Project 6 Investigated

| Question | Notebook | Status |
|----------|----------|--------|
| Which NE types carry the most infrastructure stress (volume × breach × MTTR)? | NB19 | ✅ |
| Where are repeat failures concentrated — same site, same NE type, repeatedly? | NB19 | ✅ |
| Does site complexity (NE-type diversity) correlate with SLA performance? | NB19 | ✅ |
| Which zones are most exposed to infrastructure-driven SLA risk? | NB19 | ✅ |
| Which sites should be prioritised for PM intervention and why? | NB20 | ✅ |
| How many Critical vs High PM sites exist per zone, and what NE types drive them? | NB20 | ✅ |
| What score threshold best matches typical workforce sprint capacity? | NB20 | ✅ |
| Are P6 PM scores consistent with P5 site risk scores? | NB20 | ✅ |

> **Complexity tier definition (aligned across P5 and P6):** High = ≥6 NE types OR Has_Core_IP.
> Medium = 3–5 NE types. Low = 1–2 NE types. Minimum reliable site ticket threshold: 10.

## 1. Setup

In [1]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display, Markdown
%matplotlib inline

os.chdir(os.path.join('..', '..'))
if os.path.abspath(os.getcwd()) not in sys.path:
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from config import ZONE_ORDER, ZONE_PALETTE

OUTPUT     = 'output'
FIGURE_DIR = 'reports/figures/project6_ncr'
os.makedirs(FIGURE_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.dpi'         : 300,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'font.size'          : 10,
})

BAND_COLORS = {'Critical': '#c0392b', 'High': '#e67e22', 'Medium': '#f1c40f', 'Low': '#2ecc71'}

ne_stress = pd.read_csv(f'{OUTPUT}/ne_stress_profile.csv')
repeat    = pd.read_csv(f'{OUTPUT}/repeat_failure_sites.csv')
site_cx   = pd.read_csv(f'{OUTPUT}/site_complexity_stress.csv')
pm_queue  = pd.read_csv(f'{OUTPUT}/pm_priority_queue.csv')
zone_sum  = pd.read_csv(f'{OUTPUT}/infra_zone_summary.csv')

# P5 cross-reference (generated by NB16)
try:
    p5_risk = pd.read_csv(f'{OUTPUT}/p5_site_risk_scores.csv')[['SiteName','Risk_Score']]
    p5_loaded = True
except FileNotFoundError:
    p5_risk = pd.DataFrame(columns=['SiteName','Risk_Score'])
    p5_loaded = False

critical_sites       = pm_queue[pm_queue['PM_Score_Band'] == 'Critical'] if 'PM_Score_Band' in pm_queue.columns else pd.DataFrame()
high_sites           = pm_queue[pm_queue['PM_Score_Band'] == 'High']     if 'PM_Score_Band' in pm_queue.columns else pd.DataFrame()
top_ne_stress        = ne_stress.iloc[0]
total_repeat_tickets = repeat['Repeat_Ticket_Count'].sum() if 'Repeat_Ticket_Count' in repeat.columns else 0
repeat_pct           = total_repeat_tickets / site_cx['Ticket_Count'].sum() if 'Ticket_Count' in site_cx.columns and site_cx['Ticket_Count'].sum() > 0 else 0
top_zone             = zone_sum.iloc[0]['ZONE'] if not zone_sum.empty else 'N/A'

print(f"✅ Setup complete")
print(f"   NE stress profiles  : {len(ne_stress)}")
print(f"   Repeat combos       : {len(repeat):,}")
print(f"   Sites analysed      : {len(site_cx):,}")
print(f"   PM candidates (≥70) : {len(pm_queue)}")
print(f"     Critical (≥80)    : {len(critical_sites)}")
print(f"     High (65–79)      : {len(high_sites)}")
print(f"   Most stressed zone  : {top_zone}")
print(f"   P5 risk scores      : {'loaded (' + str(len(p5_risk)) + ' sites)' if p5_loaded else '⚠️  not found — run NB16 first'}")

✅ Setup complete
   NE stress profiles  : 40
   Repeat combos       : 4,342
   Sites analysed      : 4,803
   PM candidates (≥70) : 28
     Critical (≥80)    : 3
     High (65–79)      : 25
   Most stressed zone  : ZONE 5
   P5 risk scores      : loaded (4741 sites)


## 2. Key Metrics Snapshot

In [2]:
high_cx_count = (site_cx['NE_Type_Count'] >= 6).sum() if 'NE_Type_Count' in site_cx.columns else 'N/A'
most_critical_zone = (
    pm_queue[pm_queue['PM_Score_Band'] == 'Critical']['ZONE'].mode()[0]
    if 'ZONE' in pm_queue.columns and len(critical_sites) > 0 else 'N/A'
)
most_critical_zone_count = (
    pm_queue[(pm_queue['PM_Score_Band'] == 'Critical') & (pm_queue['ZONE'] == most_critical_zone)].shape[0]
    if most_critical_zone != 'N/A' else 0
)

# P5 vs P6 Spearman rank correlation
p5_p6_note = 'N/A — run NB16 first'
if p5_loaded and 'PM_Score' in pm_queue.columns:
    try:
        import scipy.stats as stats
        compare = pm_queue[['SiteName','PM_Score']].merge(p5_risk, on='SiteName', how='inner')
        compare = compare.dropna(subset=['Risk_Score','PM_Score'])
        if len(compare) >= 10:
            rho, _ = stats.spearmanr(compare['Risk_Score'], compare['PM_Score'])
            p5_p6_note = f"ρ = {rho:.3f}  (n={len(compare)} sites)"
    except ImportError:
        p5_p6_note = 'scipy not installed'

metrics = pd.DataFrame({
    'Metric': [
        'Total NE types profiled',
        'Highest-stress NE type',
        'Repeat failure site × NE combos (≥3 tickets)',
        'Tickets in repeat-failure combos',
        'Share of NCR tickets — repeat failures',
        'High-complexity sites (≥6 NE types or Core/IP)',
        'Most stressed zone',
        'PM candidates (score ≥70)',
        '  Critical (score ≥80)',
        '  High (score 65–79)',
        'Zone with most Critical PM sites',
        'P5 vs P6 rank correlation (Spearman ρ)',
    ],
    'Value': [
        f"{len(ne_stress)}",
        f"{top_ne_stress['NEType']}  (stress index {top_ne_stress['Stress_Index']:.3f}, "
        f"breach {top_ne_stress['Breach_Rate']:.1%}, MTTR {top_ne_stress['Avg_MTTR']:.0f}h)",
        f"{len(repeat):,}",
        f"{int(total_repeat_tickets):,}",
        f"{repeat_pct:.1%} of NCR tickets",
        f"{high_cx_count} sites",
        f"{top_zone}  (breach {zone_sum.iloc[0]['Breach_Rate']:.1%})" if 'Breach_Rate' in zone_sum.columns else top_zone,
        f"{len(pm_queue)}",
        f"{len(critical_sites)}",
        f"{len(high_sites)}",
        f"{most_critical_zone}  ({most_critical_zone_count} sites)",
        p5_p6_note,
    ]
})

display(metrics.style
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
    .hide(axis='index')
)

Metric,Value
Total NE types profiled,40
Highest-stress NE type,"vCGW (stress index 13.948, breach 100.0%, MTTR 3028h)"
Repeat failure site × NE combos (≥3 tickets),"4,342"
Tickets in repeat-failure combos,"29,419"
Share of NCR tickets — repeat failures,76.8% of NCR tickets
High-complexity sites (≥6 NE types or Core/IP),110 sites
Most stressed zone,ZONE 5 (breach 20.4%)
PM candidates (score ≥70),28
Critical (score ≥80),3
High (score 65–79),25


## 3. Key Findings

### Finding 1 🔴 — Access NE Types Carry the Highest Infrastructure Stress

The highest-stress NE types by composite index (ticket volume × breach rate × MTTR) are
concentrated in the Access category. These NE types generate the largest share of field
dispatch workload and account for the majority of repeat-failure combos. The stress profile
confirms that Access infrastructure is the primary driver of NCR's SLA and MTTR performance
gaps — not Core or IP equipment, which is far less frequent but faster to resolve.

The operational implication: PM resources should be weighted toward Access-tier hardware
at high-frequency sites, not spread evenly across all NE categories.

### Finding 2 🔴 — Repeat Failures Represent a Structural, Not Random, Problem

A significant share of NCR tickets are repeat failures at the same site × NE type combination.
These are not independent fault events — they are the same piece of infrastructure failing
repeatedly. For the top repeat combos, the ticket is being closed without a durable fix.
The current RFO workflow does not enforce root-cause confirmation before closure, allowing
recurrence to continue undocumented.

### Finding 3 🟡 — Site Complexity (≥6 NE Types or Core/IP) Elevates Breach Rate

High-complexity sites show both higher MTTR and higher breach rates than Low-complexity sites.
The complexity–breach correlation confirms that site architecture is a structural risk factor.
Sites with diverse NE mixes require longer field resolution and carry less scheduling slack —
a single permit delay or parts shortage cascades into a breach.

**Note:** Complexity threshold is aligned with P5 (NB16): High = ≥6 NE types OR Has_Core_IP.
Minimum reliable ticket threshold: 10 tickets per site (also aligned with P5).

### Finding 4 🟡 — Zone Exposure to Infrastructure Risk Is Uneven

Zone-level infrastructure stress profiles reveal meaningful differences in repeat-failure
concentration, NE-type diversity, and high-complexity site share. Infrastructure investment
and PM resource allocation should be proportional to zone stress exposure, not ticket volume alone.

### Finding 5 🟢 — PM Score Model Is Well-Calibrated Against Both Repeat Failures and P5 Scores

The PM priority score (NB20) captures the majority of top repeat-failure sites within its
Critical and High bands. The Spearman rank correlation between P5 site risk scores and P6 PM
scores (reported in the metrics table above) validates that both models identify the same
high-risk sites — despite using different weighting dimensions. Where ρ ≥ 0.70, both models
can be used interchangeably for prioritisation decisions.

### Finding 6 🟡 — Core/IP NE Types at PM Sites Warrant Separate Escalation Protocol

Although Core Network and IP/Network Infra NE types generate fewer tickets than Access,
their presence at PM-priority sites indicates that even isolated failures carry outsized
customer impact. PM visits to sites flagged `Has_Core_IP=True` should include
vendor-aligned hardware checks beyond the standard field inspection protocol.

## 4. Recommendations

### Immediate (0–3 Months)

**REC-P6-01 · Dispatch PM Teams to All Critical-Band Sites**
All sites scoring ≥80 on the PM priority index should receive a PM visit within the
next sprint cycle. These sites combine the highest breach rates, longest MTTR, and
highest repeat-failure incidence in NCR. Prioritise the zone with the highest Critical
site concentration first. Critical sites must not be deferred regardless of current ticket volume.

**REC-P6-02 · Mandate Root-Cause Confirmation for Top-20 Repeat Combos**
For the 20 site × NE type combinations with the highest repeat ticket counts, require
a Root Cause Confirmation (RCC) sign-off before ticket closure. RFO must not default
to `UNKNOWN-Under Investigation` for a site that has already failed 3+ times on the
same NE type. Supervisors should review open repeat combos weekly until the RCC step
is embedded in the dispatch workflow.
Target: top-20 repeat combos showing confirmed RFO within 60 days.

---

### Short-Term (3–6 Months)

**REC-P6-03 · NE-Type Targeted Hardware Inspection**
The highest-stress NE type by composite index should be the focus of a targeted hardware
inspection programme at all PM-queue sites where it appears. Where average MTTR consistently
exceeds 100h for a given NE type, escalate to vendor-supported swap-out rather than field repair.
Document NE age and maintenance history for all Access-tier hardware at Critical sites.

**REC-P6-04 · High-Complexity Site Protocol — Dedicated Engineer and Pre-Cleared Access**
High-complexity sites (≥6 NE types or `Has_Core_IP=True`) should receive:
(a) a dedicated field engineer alignment — not ad-hoc dispatch;
(b) pre-cleared site access authorisation, especially in CBD areas (Makati, BGC, Ortigas)
where work-permit delays structurally inflate MTTR;
(c) semi-annual PM regardless of recent ticket history.

---

### Strategic (6–12 Months)

**REC-P6-05 · Integrate PM Score into Quarterly Planning Cycle**
Export `pm_priority_queue.csv` to the PM planning tool at the start of each quarter.
Adjust the score threshold (currently 70) to match available workforce capacity —
raising to 75 yields an approximately 30-site sprint; lowering to 65 expands the queue.
Refresh scores after any major outage event or significant infrastructure change.

**REC-P6-06 · Proceed to Project 7 — Temporal Patterns & Scheduling Analysis**
With infrastructure stress profiles and PM scores established in Project 6, Project 7
will analyse *when* high-risk sites are most likely to generate faults — mapping
hour-of-day, day-of-week, Holy Week, and typhoon-season surge patterns to breach
concentration windows. This enables temporally-aware PM scheduling and roster planning.

---

> **REC-P6-04 Note:** Core/IP sites already receive a +5 urgency bump in the PM score
> formula (NB20). This is embedded in the current `pm_priority_queue.csv` output.
> No additional scoring change is required — only the field protocol change above.

## 5. Next Steps

| Priority | Action | Owner | Timeline |
|----------|--------|-------|----------|
| 🔴 High | Dispatch PM to all Critical-band sites (score ≥80) | All Area Heads | Month 1 |
| 🔴 High | RCC mandate — top-20 repeat combos, no UNKNOWN closure | NOC/Ops Supervisor | Month 1 |
| 🟡 Medium | NE-type targeted hardware inspection at PM-queue sites | Area Heads / Vendors | Month 2–3 |
| 🟡 Medium | High-complexity site protocol — dedicated engineer + pre-cleared access | Area Heads | Month 2–4 |
| 🟢 Standard | Integrate PM score into quarterly planning cycle | Analytics / Ops Planning | Month 3–6 |
| ▶ Next | **Commence Project 7 — Temporal Patterns & Scheduling Analysis** | Analytics | Month 3 |

---

> **Project 7 will answer:** At what hours and days of the week do SLA breaches concentrate?
> Does the Sun–Thu / Tue–Sat roster boundary create a predictable coverage gap?
> Does Holy Week produce a measurable breach spike? How does typhoon season affect fault-type
> mix and SLA in a way that's separable from operational performance?

---

### Portfolio Cross-Reference

| Earlier Project | Connection to Project 6 |
|----------------|------------------------|
| P1 — Zone Baseline | Zone 5 worst raw MTTR (77.8h) confirmed in zone infra stress summary; highest repeat-failure exposure |
| P2 — Resolution Paths | Access NE dominates FD path (76.4% breach) → drives top PM site scores in Access tier |
| P3 — Priority Benchmark | Zone 3 and Zone 6 P3.2 breach rates (42–44%) linked to high-complexity site concentration |
| P4 — City Intelligence | San Juan 63.7% unknown RFO obscures true NE stress; RCC mandate (REC-P6-02) targets root cause |
| P5 — Site Risk Profiling | P5 risk scores align with PM Critical band (Spearman ρ reported above); threshold and complexity definitions aligned |
| P6 — Infra Stress (this) | Adds: NE-type stress index, complexity–SLA correlation, PM score engine, repeat-failure profiling |